In [12]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.utils import resample
from joblib import Parallel, delayed
import multiprocessing

##############################
point_for_prediction = 0.05

# Simulation parameters
n = 500
B = 500        # Paper: 10_000
nsim = 1000    # Paper:  1_000
seed = 123

##############################

# Define the theta function
def theta(x, y):
    model = DecisionTreeRegressor(max_leaf_nodes=5)
    model.fit(x, y)
    return model.predict([[point_for_prediction]])[0], model

# Define the bootstrap function
def bootstrap(indices, B, theta, x, y):
    thetastar = np.empty(B)
    models = []
    for b in range(B):
        sample_indices = resample(indices, replace=True)
        sample_x = x[sample_indices]
        sample_y = y[sample_indices]
        thetastar[b], m = theta(sample_x, sample_y)
        models.append(m)
    return thetastar, models

# Set up parallel processing
num_cores = multiprocessing.cpu_count() - 1

# Main simulation loop
def simulate(sim):
    np.random.seed(sim + seed)  # For reproducibility
    x1 = np.random.uniform(0, 1, n)
    e = np.random.normal(0, np.sqrt(2), n)
    
    y = (0.7 * ((x1 >= 0.35) & (x1 < 0.45)) +
         0.7 * ((x1 >= 0.55) & (x1 < 0.65)) +
         1.4 * ((x1 >= 0.45) & (x1 < 0.55)) +
         e)
    
    x1 = x1.reshape(-1, 1)  # Reshape for sklearn
    indices = np.arange(n)
    
    p, models_sims = bootstrap(indices, B, theta, x1, y)
    return np.mean(p), models_sims

# Run simulations in parallel
results = Parallel(n_jobs=num_cores)(delayed(simulate)(sim) for sim in range(nsim))

# Separate the results into means and models
means = [res[0] for res in results]
models = [res[1] for res in results]

# Calculate the summary statistics
mean_z = np.mean(means)
var_z = np.var(means)

# Output the results
print(f"Mean: {mean_z}")
print(f"Variance: {var_z}")

Mean: -0.011507761896979673
Variance: 0.02948913608548415
First model: DecisionTreeRegressor(max_leaf_nodes=5)


In [ ]:
# Further analysis on models if needed
# Example: Analyzing the first model of the first simulation
first_model = models[0][0]
print(f"First model: {first_model}")

# plotte mir die Bäume
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(12, 12))
plot_tree(first_model, ax=ax)
plt.show()